# Walk-forward validation

Uses generated synthetic prices for software research only; these are not actual market data.

Capital resets per fold. Reported mean fold return is an arithmetic summary, not a compounded portfolio return. Test folds are held out from each run.


In [ ]:
from pathlib import Path
import sys
repo = Path.cwd()
while not (repo / "backend").is_dir() and repo != repo.parent:
    repo = repo.parent
sys.path.insert(0, str(repo / "backend"))

import pandas as pd
from trading.backtesting.walk_forward import WalkForwardConfig, run_walk_forward
from trading.data.synthetic import generate_synthetic_ohlcv
from trading.features.feature_pipeline import engineer_features
from trading.strategies.moving_average import MovingAverageCrossoverStrategy
bars = engineer_features(generate_synthetic_ohlcv(periods=1000, frequency="1D", seed=31))
result = run_walk_forward(
    bars, lambda training, validation: MovingAverageCrossoverStrategy(),
    config=WalkForwardConfig(training_window=300, validation_window=100,
                             test_window=100, step_size=100))
rows = [{"fold": fold.fold, "test_start": fold.test_start, "test_end": fold.test_end,
         "return": fold.metrics["total_return"], "drawdown": fold.metrics["max_drawdown"],
         "sharpe": fold.metrics["sharpe_ratio"],
         "buy_hold_return": fold.comparison["buy_and_hold"]["total_return"]}
        for fold in result.folds]
fold_table = pd.DataFrame(rows)
returns = pd.Series([fold.metrics["total_return"] for fold in result.folds])
fold_table, {"folds": len(returns), "mean_return": returns.mean(),
             "median_return": returns.median(),
             "profitable_rate": (returns > 0).mean(), "worst_return": returns.min()}
